# Test Data (30%) - Cafe Location Suitability Model
This notebook loads and displays the **test dataset** containing 30% of all cafe records used for model evaluation.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Set data directory
DATA_DIR = '../data'

print("Loading all 8 datasets...")
amenities_df = pd.read_csv(os.path.join(DATA_DIR, 'amenities_clean.csv'))
features_df = pd.read_csv(os.path.join(DATA_DIR, 'dataset_ft_enriched.csv'))
cafes_df = pd.read_csv(os.path.join(DATA_DIR, 'kathmandu_cafes.csv'))  # PRIMARY
census_df = pd.read_csv(os.path.join(DATA_DIR, 'kathmandu_census.csv'))
education_df = pd.read_csv(os.path.join(DATA_DIR, 'kathmandu_education_cleaned.csv'))
osm_amenities_df = pd.read_csv(os.path.join(DATA_DIR, 'osm_amenities_kathmandu.csv'))
roads_df = pd.read_csv(os.path.join(DATA_DIR, 'osm_roads_kathmandu.csv'))
wards_df = pd.read_csv(os.path.join(DATA_DIR, 'kathmandu_wards_boundary_sorted.csv'))

print(f"✓ All datasets loaded successfully")
print(f"Primary dataset: {len(cafes_df):,} cafes")

Loading all 8 datasets...


FileNotFoundError: [Errno 2] No such file or directory: '../data\\amenities_clean.csv'

In [ ]:
# Load pre-computed features and target from the main training notebook
# These should have been saved as pickle files
import pickle

# Load features (X) and target (y)
try:
    with open('../data/X_features.pkl', 'rb') as f:
        X = pickle.load(f)
    with open('../data/y_target.pkl', 'rb') as f:
        y = pickle.load(f)
    print(f"✓ Loaded pre-computed features and target")
except FileNotFoundError:
    print("Features/target files not found. Using features_df directly...")
    # Use features_df as basis
    X = features_df.drop(['cafe_id', 'name', 'suitability_score'], axis=1, errors='ignore')
    y = features_df['suitability_score'] if 'suitability_score' in features_df.columns else None

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape if hasattr(y, 'shape') else len(y)}")

In [ ]:
# Perform 70/30 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30,  # 30% for testing
    random_state=42
)

print("="*80)
print("TRAIN-TEST SPLIT SUMMARY (70/30 Strategy)")
print("="*80)
print(f"\nTotal samples:     {len(X):,}")
print(f"Training samples:  {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test samples:      {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nFeatures per sample: {X_test.shape[1]}")
print(f"Random seed: 42 (for reproducibility)")
print(f"\n✓ Test data shape: {X_test.shape}")
print(f"✓ Test target shape: {y_test.shape}")

## TEST DATA (30% - {len(X_test):,} samples)

In [ ]:
# Display first 10 test samples
print("\n" + "="*80)
print("FIRST 10 TEST SAMPLES - FEATURES (X_test)")
print("="*80)
display(X_test.head(10))

print("\nFirst 10 test target values (y_test):")
print(y_test.head(10).values)

In [ ]:
# Test data statistics
print("\n" + "="*80)
print("TEST DATA STATISTICS")
print("="*80)

print("\nTarget Variable (y_test) Statistics:")
print(f"  Count:     {y_test.count()}")
print(f"  Mean:      {y_test.mean():.4f}")
print(f"  Std Dev:   {y_test.std():.4f}")
print(f"  Min:       {y_test.min():.4f}")
print(f"  25%:       {y_test.quantile(0.25):.4f}")
print(f"  Median:    {y_test.median():.4f}")
print(f"  75%:       {y_test.quantile(0.75):.4f}")
print(f"  Max:       {y_test.max():.4f}")

print("\nFeature Matrix (X_test) Statistics:")
feature_stats = X_test.describe().T
display(feature_stats)

In [ ]:
# Compare training vs test distributions
print("\n" + "="*80)
print("TRAIN vs TEST DISTRIBUTION COMPARISON")
print("="*80)

comparison = pd.DataFrame({
    'Metric': ['Count', 'Mean', 'Std Dev', 'Min', 'Max', '25%', '75%'],
    'Training (70%)': [
        y_train.count(),
        f"{y_train.mean():.4f}",
        f"{y_train.std():.4f}",
        f"{y_train.min():.4f}",
        f"{y_train.max():.4f}",
        f"{y_train.quantile(0.25):.4f}",
        f"{y_train.quantile(0.75):.4f}"
    ],
    'Test (30%)': [
        y_test.count(),
        f"{y_test.mean():.4f}",
        f"{y_test.std():.4f}",
        f"{y_test.min():.4f}",
        f"{y_test.max():.4f}",
        f"{y_test.quantile(0.25):.4f}",
        f"{y_test.quantile(0.75):.4f}"
    ]
})
display(comparison)

In [ ]:
# Data distribution visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Target distribution
axes[0, 0].hist(y_test, bins=30, color='salmon', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Test Target Distribution (Suitability Score)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Suitability Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

# Box plot of target
axes[0, 1].boxplot(y_test, vert=True)
axes[0, 1].set_title('Test Target Box Plot', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Suitability Score')
axes[0, 1].grid(alpha=0.3)

# Feature means
feature_means = X_test.mean()
axes[1, 0].barh(range(len(feature_means)), feature_means.values, color='lightblue', edgecolor='black')
axes[1, 0].set_yticks(range(len(feature_means)))
axes[1, 0].set_yticklabels(feature_means.index, fontsize=9)
axes[1, 0].set_title('Mean Feature Values (Test Data)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Mean Value')
axes[1, 0].grid(alpha=0.3, axis='x')

# Missing values
missing_counts = X_test.isnull().sum()
axes[1, 1].barh(range(len(missing_counts)), missing_counts.values, color='plum', edgecolor='black')
axes[1, 1].set_yticks(range(len(missing_counts)))
axes[1, 1].set_yticklabels(missing_counts.index, fontsize=9)
axes[1, 1].set_title('Missing Values per Feature (Test Data)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Count')
axes[1, 1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/test_data_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to ../outputs/test_data_overview.png")

In [ ]:
# Side-by-side distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training distribution
axes[0].hist(y_train, bins=30, color='skyblue', edgecolor='black', alpha=0.7, label='Training (70%)')
axes[0].axvline(y_train.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {y_train.mean():.4f}')
axes[0].set_title('Training Data Distribution (70%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Suitability Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Test distribution
axes[1].hist(y_test, bins=30, color='salmon', edgecolor='black', alpha=0.7, label='Test (30%)')
axes[1].axvline(y_test.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {y_test.mean():.4f}')
axes[1].set_title('Test Data Distribution (30%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Suitability Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/train_test_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Train-Test comparison saved to ../outputs/train_test_comparison.png")

In [ ]:
# Export test data
print("\n" + "="*80)
print("EXPORTING TEST DATA")
print("="*80)

# Save as CSV
X_test.to_csv('../data/test_features_30percent.csv', index=False)
y_test.to_csv('../data/test_target_30percent.csv', index=False)

# Save as pickle (maintains data types)
import pickle
with open('../data/test_X.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open('../data/test_y.pkl', 'wb') as f:
    pickle.dump(y_test, f)

print(f"✓ Saved test features to test_features_30percent.csv ({X_test.shape})")
print(f"✓ Saved test target to test_target_30percent.csv ({y_test.shape})")
print(f"✓ Saved test data to pickle files (test_X.pkl, test_y.pkl)")
print(f"\n🎯 TEST DATA (30%) IS READY FOR MODEL EVALUATION")